In [ ]:
%pip install -e ../../../

In [ ]:
import numpy as np
from numpy.random import default_rng
RNG = default_rng(12814)
import pandas as pd

# We have history of inlfation from 1999 because water prices in 2000 are calculated
# as water price in 1999 by inflation in 1999
HISTORICAL_YEARS = pd.date_range(start='1999-01-01', end='2024-01-01', freq='YS')

## Inflation

We upload the inflation history and predict it for the future.

In [ ]:
inflation_df = pd.read_excel(
    '../raw-data/Inflation_Data.xlsx',
    header=3,
    index_col=0,
    names=['timestamp', 'NL0000']
    )
inflation_df = inflation_df.iloc[:69, :]
inflation_df.index = pd.to_datetime(inflation_df.index, format='%Y')
inflation_df['NL0000'] = inflation_df['NL0000'].astype(float)

history_inflation = inflation_df.loc[HISTORICAL_YEARS, :] # from 2000 to 2024

BCE_TARGET_INFLATION = 2 # %
# Generate range from 2025 to 2035, every 5 years (start of year)
years = pd.date_range(start='2025-01-01', end='2035-01-01', freq='5YS')
inflation_df_out = history_inflation.copy()
for year in years:
    inflation_df_out.loc[year, 'NL0000'] = BCE_TARGET_INFLATION

inflation_df_out.index.name = 'timestamp'

## Bonds

The two exogenous variables are:

- inflationary expectations (base credit spread influencing the coupon) 
- investors' demand

In [ ]:
# Naive proxy of inflationary expectations: previous 3 years average
history_inflationary_expectations = (
    inflation_df
    .shift(1) # Shift to use history
    .rolling(window=3).mean()
    .loc[HISTORICAL_YEARS, :]
)
history_inflationary_expectations.index.name = 'timestamp'

In [ ]:
# Investor demand moves between 0.6 and 0.4
MIN_INVESTOR_DEMAND = 0.6
MAX_INVESTOR_DEMAND = 1.4
invdemand_df = pd.DataFrame(
    data={'NL0000': RNG.uniform(
        low=MIN_INVESTOR_DEMAND,
        high=MAX_INVESTOR_DEMAND,
        size=(len(HISTORICAL_YEARS),)
    )},
    index=HISTORICAL_YEARS
)
invdemand_df.index.name = 'timestamp'

In [ ]:
# Save out
from water_futures_battle.core.base_model import DynamicProperties
econo_dps = DynamicProperties('economy-dynamic_properties')

econo_dps['inflation'] = inflation_df_out.round(2)
econo_dps['inflation-expect'] = history_inflationary_expectations.round(3)
econo_dps['investor_demand'] = invdemand_df.round(4)
econo_dps.dump(path='..')